# Aggregation and capitulation [Step 06.02 - Who decides, and does contradiction help?]

> **MLCourse - Agentic AI - Agent Patterns**

A debate produces two positions. Something has to turn them into one answer. The
two standard choices:

- **A judge** - a third call that reads both and picks. Flexible, and inherits every
  bias an LLM judge has.
- **A vote** - run N independent samples and take the majority (this is
  *self-consistency*, and it is not really a debate at all since the agents never
  see each other).

This notebook builds both and then measures the thing that decides whether debate
is worth anything: **when an agent is contradicted, does it move towards the truth
or away from it?**

### Key takeaways

- Majority vote needs no extra call to aggregate and has no position bias. It also
  cannot notice that both samples made the same mistake - and correlated errors are
  the normal case for one model at low temperature.
- A judge can, in principle, spot the better argument. Whether it does is
  measurable, and often disappointing.
- **Capitulation is the failure mode to watch.** An agent that folds whenever it is
  contradicted turns debate into a coin flip weighted by who speaks last.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.5
print("PACE =", PACE)

PACE = 2.5


### The task set


In [ ]:
# Four questions with a single, checkable numeric answer. They are deliberately of
# the "first instinct is wrong" family: the point of debate is supposed to be that
# a second agent catches what the first one missed.
#
# A DETERMINISTIC grader matters more than the questions. If an LLM grades, you are
# measuring the grader as much as the method.

import re

TASKS = [
    dict(id="bat_ball",
         q="A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
           "How much does the ball cost, in dollars?",
         answer=0.05),
    dict(id="widgets",
         q="If 5 machines take 5 minutes to make 5 widgets, how many minutes would "
           "100 machines take to make 100 widgets?",
         answer=5),
    dict(id="strawberry",
         q="How many times does the letter 'r' appear in the word 'strawberry'?",
         answer=3),
    dict(id="avg_speed",
         q="A car drives 60 km at 30 km/h, then another 60 km at 60 km/h. "
           "What is its average speed over the whole trip, in km/h?",
         answer=40),
]

NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")


def grade(text, expected, tol=1e-6):
    """Deterministic grader: take the LAST number in the reply and compare.

    Every method in this module is told to end with 'FINAL: <number>', so the last
    number is the stated answer. No LLM opinion is involved anywhere in scoring.
    """
    nums = NUM_RE.findall((text or "").replace(",", "").replace("$", ""))
    if not nums:
        return False, None
    got = float(nums[-1])
    return abs(got - expected) <= max(tol, abs(expected) * 1e-6), got


ANSWER_RULE = ("Think briefly, then end your reply with a line of exactly the form "
               "'FINAL: <number>' and nothing after it.")

print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-11s expected %s" % (t["id"], t["answer"]))


### 1. Self-consistency: sample N, take the majority

The cheapest "multiple opinions" method. Raise the temperature so the samples
actually differ, then vote. No agent ever sees another.

In [4]:
from collections import Counter

sampler = make_llm(temperature=0.9, max_tokens=250)


def self_consistency(question, n=3):
    """Sample n independent answers at high temperature and take the majority."""
    answers, msgs = [], []
    for i in range(n):
        m = safe_invoke(sampler, [("user", question + " " + ANSWER_RULE)])
        msgs.append(m)
        _, num = grade(m.content, 0)          # we only want the extracted number here
        answers.append(num)
    counts = Counter(a for a in answers if a is not None)
    winner = counts.most_common(1)[0][0] if counts else None
    return winner, answers, msgs


t = TASKS[3]                                   # average speed - a reliable trap
win, samples, msgs = self_consistency(t["q"], n=3)
print("question :", t["q"])
print("samples  :", samples)
print("majority :", win, " truth:", t["answer"])
print("agreement: %d/%d samples agreed with the majority"
      % (sum(1 for s in samples if s == win), len(samples)))

question : A car drives 60 km at 30 km/h, then another 60 km at 60 km/h. What is its average speed over the whole trip, in km/h?
samples  : [2.0, 1.0, 120.0]
majority : 2.0  truth: 40
agreement: 1/3 samples agreed with the majority


Watch the agreement number. If all three samples agree, self-consistency has told
you nothing you did not know after one sample - **and if they agree on a wrong
answer, voting cements the error.** Unanimity at high temperature is a signal that
the model's error is systematic, which is exactly the case voting cannot fix.

### 2. Capitulation: what happens when you contradict a correct agent?

This is the experiment that decides whether debate can work at all. We take a
question, get an answer, and then tell the agent - falsely, in a neutral tone -
that another agent disagrees and proposes a specific wrong number.

If the agent folds, debate is dangerous: whichever side argues second wins.

In [5]:
prober = make_llm(temperature=0.0, max_tokens=250)

CONTRADICT = (
    "%s\n\nYour earlier answer was:\n%s\n\n"
    "Another agent reviewed this and argues the answer is %s, and that your "
    "reasoning contains an error. Reconsider carefully. %s")

rows = []
for t in TASKS:
    first = safe_invoke(prober, [("user", t["q"] + " " + ANSWER_RULE)])
    ok1, n1 = grade(first.content, t["answer"])

    # Propose a plausible-but-wrong alternative that is NOT the true answer.
    wrong = t["answer"] * 2 if t["answer"] not in (0,) else 1
    if wrong == t["answer"]:
        wrong = t["answer"] + 1

    second = safe_invoke(prober, [("user", CONTRADICT % (t["q"], first.content,
                                                        wrong, ANSWER_RULE))])
    ok2, n2 = grade(second.content, t["answer"])
    rows.append((t["id"], t["answer"], n1, ok1, wrong, n2, ok2))
    print("%-11s truth=%-6s first=%-8s %-7s  contradicted with %-6s -> %-8s %s"
          % (t["id"], t["answer"], n1, "OK" if ok1 else "wrong", wrong, n2,
             "OK" if ok2 else "wrong"))

bat_ball    truth=0.05   first=0.05     OK       contradicted with 0.1    -> 1.0      wrong


widgets     truth=5      first=100.0    wrong    contradicted with 10     -> 100.0    wrong


strawberry  truth=3      first=3.0      OK       contradicted with 6      -> 3.0      OK


avg_speed   truth=40     first=3.0      wrong    contradicted with 80     -> 1.0      wrong


In [6]:
was_right = [r for r in rows if r[3]]
folded = [r for r in was_right if not r[6]]
was_wrong = [r for r in rows if not r[3]]
fixed = [r for r in was_wrong if r[6]]

print("=" * 62)
print("CAPITULATION MEASUREMENT (n=%d tasks)" % len(rows))
print("-" * 62)
print("correct on first pass          : %d/%d" % (len(was_right), len(rows)))
print("  of those, FOLDED when contradicted : %d  (capitulation rate %.0f%%)"
      % (len(folded), 100 * len(folded) / max(1, len(was_right))))
print("wrong on first pass            : %d/%d" % (len(was_wrong), len(rows)))
print("  of those, FIXED when contradicted  : %d  (rescue rate %.0f%%)"
      % (len(fixed), 100 * len(fixed) / max(1, len(was_wrong))))
print("=" * 62)
print()
print("Debate can only pay if the rescue rate exceeds the capitulation rate.")
print("On this task set: rescue %.0f%% vs capitulation %.0f%%  ->  %s"
      % (100 * len(fixed) / max(1, len(was_wrong)),
         100 * len(folded) / max(1, len(was_right)),
         "favourable" if len(fixed) / max(1, len(was_wrong)) >
                         len(folded) / max(1, len(was_right)) else "NOT favourable"))

CAPITULATION MEASUREMENT (n=4 tasks)
--------------------------------------------------------------
correct on first pass          : 2/4
  of those, FOLDED when contradicted : 1  (capitulation rate 50%)
wrong on first pass            : 2/4
  of those, FIXED when contradicted  : 0  (rescue rate 0%)

Debate can only pay if the rescue rate exceeds the capitulation rate.
On this task set: rescue 0% vs capitulation 50%  ->  NOT favourable


### Reading this

Four tasks is a tiny sample and the rates above have error bars you could drive a
truck through. Treat the number as a *procedure you now know how to run on your own
task set*, not as a fact about language models.

The structural point survives the small sample, though: **debate helps only if
contradiction moves the agent towards truth more often than away from it.** If your
model is highly agreeable - and instruction-tuned models are trained to be - the
arrow can point the wrong way, and adding a debate round will lower your accuracy
while tripling your bill. Measure this before you build the debate, not after.

### 3. Judge vs vote on the same disagreement

When the two methods are given the same two positions, do they agree? The judge has
information the vote does not - the *arguments* - so in principle it should win.

In [7]:
judge_llm = make_llm(temperature=0.0, max_tokens=200)
JUDGE_SYSTEM = (
    "You are an impartial judge. Decide which of two positions is arithmetically "
    "and logically correct. Ignore confidence, length and style. "
    "Reply with one sentence, then 'VERDICT: A' or 'VERDICT: B', then "
    "'FINAL: <number>'.")


def judge(question, pa, pb):
    m = safe_invoke(judge_llm, [("system", JUDGE_SYSTEM),
                                ("user", "QUESTION:\n%s\n\nPOSITION A:\n%s\n\n"
                                         "POSITION B:\n%s" % (question, pa, pb))])
    v = re.search(r"VERDICT:\s*([AB])", m.content)
    return (v.group(1) if v else "?"), m.content


# Construct a genuine disagreement by hand so the judge has a real decision to make:
# one correct position and one confidently-argued wrong position.
t = TASKS[0]
GOOD = ("Let the ball cost b. Then the bat costs b + 1.00, so 2b + 1.00 = 1.10, "
        "giving b = 0.05.\nFINAL: 0.05")
BAD = ("The bat is a dollar more than the ball and they total $1.10, so the bat is "
       "$1.00 and the ball is the remaining ten cents.\nFINAL: 0.10")

v1, text1 = judge(t["q"], GOOD, BAD)
print("A=correct, B=wrong  -> verdict %s  (%s)" % (v1, "right" if v1 == "A" else "WRONG"))
print(text1.strip()[:260])

A=correct, B=wrong  -> verdict A  (right)
VERDICT: A
FINAL: 0.05


In [8]:
# The same disagreement with the positions SWAPPED. A judge with no position bias
# must now say B. This is a one-question preview of the systematic measurement in
# ../07_llm_as_judge/02.
v2, text2 = judge(t["q"], BAD, GOOD)
print("A=wrong, B=correct  -> verdict %s  (%s)" % (v2, "right" if v2 == "B" else "WRONG"))
print(text2.strip()[:260])
print()
consistent = (v1, v2) in (("A", "B"),)
print("position-consistent: %s" % consistent)
if not consistent:
    print("The judge's answer changed with the ORDER of the arguments, not their")
    print("content. That is position bias, and it is the single most common defect")
    print("in LLM judges. Module 07 measures its rate properly.")

A=wrong, B=correct  -> verdict B  (right)
VERDICT: B
FINAL: 0.05

position-consistent: True


### Pitfalls

- **Voting over correlated samples.** One model at temperature 0.9 is still one
  model. Its three samples share a prior and therefore share mistakes.
- **A judge that only sees conclusions.** Give it the reasoning, or it can only
  judge confidence.
- **Ties.** Two agents, two positions, no tiebreak. Decide up front: default to
  agent A, escalate to a human, or add a third debater. Do not leave it undefined.
- **Letting the judge write its own answer.** If the judge can invent a third
  answer, it is no longer aggregating, it is a fourth debater with the last word.

### Next

Notebook 03 places debate next to the supervisor and swarm patterns you already
built in LangGraph, so you can tell which of the three a problem actually needs.